In [ ]:
! pip install -r requirements.txt --quiet

# Creating and Deploying an Azure AI Agent Using SDK and MCP Tools

This notebook demonstrates how to create and deploy an Azure AI Agent using the Azure SDK, with tools powered by a Model Context Protocol (MCP) server exposed through Azure API Management. It provides a programmatic approach to deploying agents that query the Halo ITSM knowledge base.

**Prerequisites:**
1. Complete the infrastructure deployment (`deploy.ps1`)
2. Create a `.env` file in this `Notebooks/` directory with the following values (see `.env.sample`):
   - `PROJECT_ENDPOINT` — Your AI Foundry project endpoint (from `terraform output ai_account_endpoint`)
   - `MODEL_DEPLOYMENT_NAME` — The model deployment name (default: `gpt-4.1`)
   - `MCP_SERVER_URL` — The APIM MCP server URL (from Azure Portal → APIM → MCP Servers)
   - `MCP_SERVER_LABEL` — A label for the MCP server (e.g., `halo-itsm-mcp`)

🔗 [How to use the Model Context Protocol tool](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/model-context-protocol-samples)


In [ ]:
from os import environ
import time
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam


load_dotenv(override=True)

## Get MCP server configuration from environment variables

In [ ]:
mcp_server_url = environ["MCP_SERVER_URL"]
mcp_server_label = environ["MCP_SERVER_LABEL"] 

## Create a project client

In [ ]:
# Create an AIProjectClient instance
project_client = AIProjectClient(
    endpoint=environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),  # Use Azure Default Credential for authentication
)

openai_client = project_client.get_openai_client()

In [ ]:
print(f"MCP URL: {mcp_server_url}")
print(f"Project endpoint: {environ['PROJECT_ENDPOINT']}")

## Initialize agent MCP tool

In [ ]:

mcp_tool = MCPTool(
        server_label=mcp_server_label,
        server_url=mcp_server_url,
        require_approval="never",  # set to "never" for smoother experience during testing
    )


## Create an agent and a thread

In [ ]:
instructions = """
You are ServiceDeskAssistant, an intelligent IT service desk support agent. Your primary role is to help users with IT support requests, incident management, and knowledge base queries.

IMPORTANT GUIDELINES:
- You MUST ONLY use the provided Halo-ITSM-MCP tools to search and retrieve information from the knowledge base
- Do NOT rely on your training data or general knowledge to answer questions
- For every user query, search the knowledge base using the available tools
- If the information is not found in the knowledge base after searching, you MUST respond with: "Unable to find in knowledge base"
- Do NOT attempt to provide answers based on general knowledge if they are not found in the knowledge base
- Always be honest about the limitations of available information in the system
- Always show the **article id**

KNOWLEDGE BASE ARTICLE HANDLING (STRICT VERBATIM RULE):
When a knowledge base article is found:
1) You MUST retrieve the FULL article body using the appropriate tool (not just a search preview).
2) You MUST output the ENTIRE article text exactly as returned by the tool.
3) You MUST NOT summarize, paraphrase, shorten, or rewrite any part of the article.
4) You MUST NOT remove any sections or metadata (title, created/edited dates, review dates, article ID, description, resolution, steps).
"""

In [ ]:


agent = project_client.agents.create_version(
    agent_name="ServiceDeskAssistantSDK",
    definition=PromptAgentDefinition(
        model=os.environ["MODEL_DEPLOYMENT_NAME"],
        instructions=instructions,
        tools=[mcp_tool],
    ),
)
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

conversation = openai_client.conversations.create()
print(f"Created conversation (id: {conversation.id})")


In [ ]:
print(f"Agent name: {agent.name}")
print(f"Agent id: {agent.id}")
print(f"Conversation id: {conversation.id}")

In [ ]:
# Send a request that triggers MCP tool calls to search the Halo ITSM knowledge base
# The MCP server exposes tools like KBArticle (search) and KBArticle/{id} (get by ID)
response = openai_client.responses.create(
    conversation=conversation.id,
    input=(
        "Remove Print Job From Queue"
    ),
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)


In [ ]:
# Handle MCP approval requests if the tool requires approval
# (Not triggered when require_approval="never", included for reference)
input_list: ResponseInputParam = []
for item in response.output:
    if item.type == "mcp_approval_request" and item.server_label == mcp_server_label and item.id:
        input_list.append(
            McpApprovalResponse(
                type="mcp_approval_response",
                approve=True,
                approval_request_id=item.id,
            )
        )

# If approvals were needed, send them back to continue execution
if input_list:
    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    )

print(f"Response: {response.output_text}")


In [ ]:
# Cleanup: Uncomment the lines below to delete the agent when you're done testing
#project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
#print("Agent deleted")
